In [13]:
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium
from pathlib import Path

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


In [14]:
# Overpass API endpoint
overpass_url = "https://overpass-api.de/api/interpreter"

# Bounding box del distrito Centro de Madrid (España)
# Formato Overpass: (sur, oeste, norte, este)
south = 40.4050
west = -3.7275
north = 40.4290
east = -3.6900

# Query con bbox global — sin nombres, así que no hay confusión con otros Madrid del mundo
query = f"""
[out:json][timeout:90][bbox:{south},{west},{north},{east}];
(
  node["amenity"="restaurant"];
  node["amenity"="bar"];
  node["amenity"="cafe"];
  node["amenity"="pub"];
  node["amenity"="fast_food"];
);
out center;
"""

print("Query preparada — Centro Madrid España")
print(f"Bounding box: sur={south}, oeste={west}, norte={north}, este={east}")

Query preparada — Centro Madrid España
Bounding box: sur=40.405, oeste=-3.7275, norte=40.429, este=-3.69


In [15]:
print("Descargando datos de OpenStreetMap... (puede tardar 30-60 segundos)")

# Añadimos headers con User-Agent para que Overpass acepte la petición
headers = {
    "User-Agent": "TFM-DataScience-JuanAlonso/1.0 (proyecto académico Máster Data Science)"
}

response = requests.post(
    overpass_url,
    data={"data": query},
    headers=headers
)

print(f"Status code: {response.status_code}")

if response.status_code == 200:
    data = response.json()
    print(f"✅ Respuesta recibida. Número de elementos: {len(data['elements'])}")
else:
    print("❌ Error en la petición. Contenido devuelto:")
    print(response.text[:500])

Descargando datos de OpenStreetMap... (puede tardar 30-60 segundos)
Status code: 200
✅ Respuesta recibida. Número de elementos: 3036


In [16]:
print(f"Status code: {response.status_code}")
print(f"Content-Type: {response.headers.get('Content-Type')}")
print("---- Primeras 500 caracteres de la respuesta ----")
print(response.text[:500])

Status code: 200
Content-Type: application/json
---- Primeras 500 caracteres de la respuesta ----
{
  "version": 0.6,
  "generator": "Overpass API 0.7.62.11 87bfad18",
  "osm3s": {
    "timestamp_osm_base": "2026-09-11T22:00:49Z",
    "copyright": "The data included in this document is from www.openstreetmap.org. The data is made available under ODbL."
  },
  "elements": [

{
  "type": "node",
  "id": 26065697,
  "lat": 40.4287093,
  "lon": -3.7019720,
  "tags": {
    "addr:city": "Madrid",
    "addr:housenumber": "7",
    "addr:postcode": "28004",
    "addr:street": "Glorieta de Bilbao",
  


In [17]:
# Ver los primeros elementos para entender qué ha devuelto
print(f"Total elementos: {len(data['elements'])}\n")

# Cuántos son 'node' y cuántos son 'way'
tipos = {}
for elem in data['elements']:
    t = elem.get('type', 'desconocido')
    tipos[t] = tipos.get(t, 0) + 1

print(f"Tipos: {tipos}\n")

# Ver los primeros 3 elementos completos
for i, elem in enumerate(data['elements'][:3]):
    print(f"--- Elemento {i+1} ---")
    print(elem)
    print()
    

Total elementos: 3036

Tipos: {'node': 3036}

--- Elemento 1 ---
{'type': 'node', 'id': 26065697, 'lat': 40.4287093, 'lon': -3.701972, 'tags': {'addr:city': 'Madrid', 'addr:housenumber': '7', 'addr:postcode': '28004', 'addr:street': 'Glorieta de Bilbao', 'amenity': 'restaurant', 'email': 'info@cafecomercialmadrid.com', 'name': 'Café Comercial', 'opening_hours': 'Mo-Th 08:30-01:00; Fr-Su 08:30-02:00', 'phone': '+34 910 88 25 25', 'website': 'https://cafecomercialmadrid.com/', 'wikidata': 'Q5017237', 'wikimedia_commons': 'Category:Café Comercial', 'wikipedia': 'es:Café Comercial'}}

--- Elemento 2 ---
{'type': 'node', 'id': 26065699, 'lat': 40.4270276, 'lon': -3.7016997, 'tags': {'addr:city': 'Madrid', 'addr:housenumber': '95', 'addr:postcode': '28004', 'addr:street': 'Calle de Fuencarral', 'amenity': 'restaurant', 'branch': 'Fuencarral', 'brand': 'Honest Greens', 'brand:wikidata': 'Q116859710', 'cuisine': 'organic', 'diet:gluten_free': 'yes', 'diet:healthy': 'yes', 'diet:vegan': 'yes', 

In [18]:
# Verificación de que estamos en Madrid España
if data['elements']:
    primer_elem = data['elements'][0]
    lat = primer_elem.get('lat')
    lon = primer_elem.get('lon')
    print(f"Primer elemento — lat: {lat}, lon: {lon}")
    
    # Madrid España está aprox en lat 40.4, lon -3.7
    if lat and 40.3 < lat < 40.5 and -3.8 < lon < -3.6:
        print("✅ Coordenadas correctas: Madrid España")
    else:
        print(f"⚠️ Coordenadas no coinciden con Madrid España")

# Además, un vistazo a los tipos de establecimiento
tipos_amenity = {}
for elem in data['elements']:
    tags = elem.get('tags', {})
    amenity = tags.get('amenity', 'sin_amenity')
    tipos_amenity[amenity] = tipos_amenity.get(amenity, 0) + 1

print("\n--- Distribución por tipo ---")
for tipo, count in sorted(tipos_amenity.items(), key=lambda x: -x[1]):
    print(f"  {tipo}: {count}")
    

Primer elemento — lat: 40.4287093, lon: -3.701972
✅ Coordenadas correctas: Madrid España

--- Distribución por tipo ---
  restaurant: 1688
  bar: 527
  cafe: 368
  fast_food: 249
  pub: 204


In [19]:
# =====================================
# DESCARGA FARMACIAS EN CENTRO MADRID
# =====================================

query_farmacias = f"""
[out:json][timeout:90][bbox:{south},{west},{north},{east}];
(
  node["amenity"="pharmacy"];
  way["amenity"="pharmacy"];
);
out center;
"""

print("Descargando farmacias...")
response_farmacias = requests.post(
    overpass_url,
    data={"data": query_farmacias},
    headers=headers
)

if response_farmacias.status_code == 200:
    data_farmacias = response_farmacias.json()
    print(f"✅ Farmacias encontradas: {len(data_farmacias['elements'])}")
else:
    print(f"❌ Error: {response_farmacias.status_code}")
    print(response_farmacias.text[:300])

Descargando farmacias...
✅ Farmacias encontradas: 149


In [22]:
# =====================================
# DESCARGA CLÍNICAS DENTALES EN CENTRO MADRID
# =====================================

query_dental = f"""
[out:json][timeout:90][bbox:{south},{west},{north},{east}];
(
  node["amenity"="dentist"];
  node["healthcare"="dentist"];
  way["amenity"="dentist"];
  way["healthcare"="dentist"];
);
out center;
"""

print("Descargando clínicas dentales...")
response_dental = requests.post(
    overpass_url,
    data={"data": query_dental},
    headers=headers
)

if response_dental.status_code == 200:
    data_dental = response_dental.json()
    print(f"✅ Clínicas dentales encontradas: {len(data_dental['elements'])}")
else:
    print(f"❌ Error: {response_dental.status_code}")
    print(response_dental.text[:300])

Descargando clínicas dentales...
✅ Clínicas dentales encontradas: 40


In [23]:
# =====================================
# ANÁLISIS COMPARATIVO DE COBERTURA
# =====================================

def analizar_cobertura(data, nombre_sector):
    """Analiza qué porcentaje de registros tiene cada campo clave."""
    total = len(data['elements'])
    if total == 0:
        return None
    
    # Contadores
    con_nombre = 0
    con_direccion = 0
    con_cp = 0
    con_telefono = 0
    con_web = 0
    con_email = 0
    con_horario = 0
    
    for elem in data['elements']:
        tags = elem.get('tags', {})
        
        if tags.get('name'):
            con_nombre += 1
        if tags.get('addr:street'):
            con_direccion += 1
        if tags.get('addr:postcode'):
            con_cp += 1
        if tags.get('phone') or tags.get('contact:phone'):
            con_telefono += 1
        if tags.get('website') or tags.get('contact:website'):
            con_web += 1
        if tags.get('email') or tags.get('contact:email'):
            con_email += 1
        if tags.get('opening_hours'):
            con_horario += 1
    
    return {
        'sector': nombre_sector,
        'total_registros': total,
        'nombre_%': round(con_nombre / total * 100, 1),
        'direccion_%': round(con_direccion / total * 100, 1),
        'cp_%': round(con_cp / total * 100, 1),
        'telefono_%': round(con_telefono / total * 100, 1),
        'web_%': round(con_web / total * 100, 1),
        'email_%': round(con_email / total * 100, 1),
        'horario_%': round(con_horario / total * 100, 1),
    }

# Analizar los 3 sectores
resultados = []
resultados.append(analizar_cobertura(data, "Hostelería"))
resultados.append(analizar_cobertura(data_farmacias, "Farmacias"))
resultados.append(analizar_cobertura(data_dental, "Clínicas dentales"))

# Mostrar como DataFrame para verlo bonito
df_cobertura = pd.DataFrame(resultados)
print("=" * 80)
print("COMPARATIVA DE COBERTURA POR SECTOR")
print("=" * 80)
df_cobertura

COMPARATIVA DE COBERTURA POR SECTOR


,sector,total_registros,nombre_%,direccion_%,cp_%,telefono_%,web_%,email_%,horario_%
0,Hostelería,3036,98.1,75.1,64.6,36.6,33.7,6.2,16.9
1,Farmacias,149,47.0,93.3,90.6,91.3,1.3,0.0,15.4
2,Clínicas dentales,40,92.5,62.5,47.5,45.0,50.0,7.5,10.0


In [24]:
# =====================================
# DESCARGA GIMNASIOS EN CENTRO MADRID
# =====================================

query_gym = f"""
[out:json][timeout:90][bbox:{south},{west},{north},{east}];
(
  node["leisure"="fitness_centre"];
  node["leisure"="sports_centre"];
  node["sport"="fitness"];
  way["leisure"="fitness_centre"];
  way["leisure"="sports_centre"];
  way["sport"="fitness"];
);
out center;
"""

print("Descargando gimnasios y centros deportivos...")
response_gym = requests.post(
    overpass_url,
    data={"data": query_gym},
    headers=headers
)

if response_gym.status_code == 200:
    data_gym = response_gym.json()
    print(f"✅ Gimnasios/centros deportivos encontrados: {len(data_gym['elements'])}")
    
    # Analizar cobertura
    resultado_gym = analizar_cobertura(data_gym, "Gimnasios")
    print(f"\nCobertura de campos:")
    for campo, valor in resultado_gym.items():
        print(f"  {campo}: {valor}")
else:
    print(f"❌ Error: {response_gym.status_code}")
    print(response_gym.text[:300])

Descargando gimnasios y centros deportivos...
✅ Gimnasios/centros deportivos encontrados: 51

Cobertura de campos:
  sector: Gimnasios
  total_registros: 51
  nombre_%: 84.3
  direccion_%: 58.8
  cp_%: 43.1
  telefono_%: 23.5
  web_%: 43.1
  email_%: 3.9
  horario_%: 7.8


In [25]:
# =====================================
# COMPARATIVA: RESTAURANTES vs BARES vs OTROS
# =====================================

# Separar por amenity
restaurantes = [e for e in data['elements'] if e.get('tags', {}).get('amenity') == 'restaurant']
bares = [e for e in data['elements'] if e.get('tags', {}).get('amenity') == 'bar']
cafes = [e for e in data['elements'] if e.get('tags', {}).get('amenity') == 'cafe']

def cobertura_lista(elementos, nombre):
    total = len(elementos)
    if total == 0:
        return None
    con_nombre = sum(1 for e in elementos if e.get('tags', {}).get('name'))
    con_dir = sum(1 for e in elementos if e.get('tags', {}).get('addr:street'))
    con_cp = sum(1 for e in elementos if e.get('tags', {}).get('addr:postcode'))
    con_tel = sum(1 for e in elementos if e.get('tags', {}).get('phone') or e.get('tags', {}).get('contact:phone'))
    con_web = sum(1 for e in elementos if e.get('tags', {}).get('website') or e.get('tags', {}).get('contact:website'))
    con_cocina = sum(1 for e in elementos if e.get('tags', {}).get('cuisine'))
    
    return {
        'tipo': nombre,
        'total': total,
        'nombre_%': round(con_nombre / total * 100, 1),
        'direccion_%': round(con_dir / total * 100, 1),
        'cp_%': round(con_cp / total * 100, 1),
        'telefono_%': round(con_tel / total * 100, 1),
        'web_%': round(con_web / total * 100, 1),
        'cocina_%': round(con_cocina / total * 100, 1),
    }

resultados = [
    cobertura_lista(restaurantes, "Restaurantes"),
    cobertura_lista(bares, "Bares"),
    cobertura_lista(cafes, "Cafeterías"),
]

df_comp = pd.DataFrame(resultados)
print("=" * 80)
print("COBERTURA POR SUBSECTOR DE HOSTELERÍA")
print("=" * 80)
df_comp

COBERTURA POR SUBSECTOR DE HOSTELERÍA


,tipo,total,nombre_%,direccion_%,cp_%,telefono_%,web_%,cocina_%
0,Restaurantes,1688,98.6,78.8,69.1,45.6,41.8,54.1
1,Bares,527,98.1,79.7,68.7,26.6,18.0,5.3
2,Cafeterías,368,96.7,68.5,52.4,23.6,29.1,25.5


In [26]:
# =====================================
# CONVERTIR RESTAURANTES A DATAFRAME
# =====================================

registros_rest = []

for elem in restaurantes:
    tags = elem.get('tags', {})
    
    registro = {
        'osm_id': elem.get('id'),
        'osm_type': elem.get('type'),
        'lat': elem.get('lat'),
        'lon': elem.get('lon'),
        'name': tags.get('name'),
        'cuisine': tags.get('cuisine'),
        'addr_street': tags.get('addr:street'),
        'addr_housenumber': tags.get('addr:housenumber'),
        'addr_postcode': tags.get('addr:postcode'),
        'addr_city': tags.get('addr:city'),
        'phone': tags.get('phone') or tags.get('contact:phone'),
        'website': tags.get('website') or tags.get('contact:website'),
        'opening_hours': tags.get('opening_hours'),
        'outdoor_seating': tags.get('outdoor_seating'),
        'takeaway': tags.get('takeaway'),
        'delivery': tags.get('delivery'),
        'wheelchair': tags.get('wheelchair'),
    }
    
    registros_rest.append(registro)

df_rest = pd.DataFrame(registros_rest)
print(f"DataFrame de restaurantes creado: {len(df_rest)} filas × {len(df_rest.columns)} columnas")
print(f"\nPrimeras 3 filas:")
df_rest.head(3)

DataFrame de restaurantes creado: 1688 filas × 17 columnas

Primeras 3 filas:


,osm_id,osm_type,lat,lon,name,cuisine,addr_street,addr_housenumber,addr_postcode,addr_city,phone,website,opening_hours,outdoor_seating,takeaway,delivery,wheelchair
0,26065697,node,40.428709,-3.701972,Café Comercial,None,Glorieta de Bilbao,7,28004,Madrid,+34 910 88 25 25,https://cafecomercialmadrid.com/,Mo-Th 08:30-01:00; Fr-Su 08:30-02:00,None,None,None,None
1,26065699,node,40.427028,-3.701700,Honest Greens,organic,Calle de Fuencarral,95,28004,Madrid,None,https://honestgreens.com/,"Mo-Th 08:30-23:00, Fr 08:30-24:00, Sa 09:30-24...",None,None,None,no
2,26808568,node,40.425766,-3.712090,La Parrilla de Nino,None,Plaza de Cristino Martos,2,28015,Madrid,+34 915 59 60 11,None,None,yes,None,None,limited


In [27]:
!pip install pyarrow

In [28]:
!pip install --upgrade pyarrow

In [29]:
# =====================================
# GUARDAR CAPA RAW
# =====================================

from pathlib import Path

# Asegurar que la carpeta existe
raw_dir = Path("../data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

# Guardar en Parquet (formato eficiente para Data Science)
ruta_parquet = raw_dir / "restaurantes_centro_madrid_osm.parquet"
df_rest.to_parquet(ruta_parquet, index=False)

# Guardar también como CSV para poder abrirlo fácil en Excel/inspección
ruta_csv = raw_dir / "restaurantes_centro_madrid_osm.csv"
df_rest.to_csv(ruta_csv, index=False)

print(f"✅ Guardado como Parquet: {ruta_parquet}")
print(f"✅ Guardado como CSV: {ruta_csv}")
print(f"\nRegistros guardados: {len(df_rest)}")
print(f"Tamaño CSV: {ruta_csv.stat().st_size / 1024:.1f} KB")
print(f"Tamaño Parquet: {ruta_parquet.stat().st_size / 1024:.1f} KB")

✅ Guardado como Parquet: ../data/raw/restaurantes_centro_madrid_osm.parquet
✅ Guardado como CSV: ../data/raw/restaurantes_centro_madrid_osm.csv

Registros guardados: 1688
Tamaño CSV: 197.7 KB
Tamaño Parquet: 110.7 KB
